# Base 0 (Zero-Shot) Inference Orchestrator

Este notebook realiza el clonado sparse del repositorio, instalación de dependencias en modo editable y orquesta la inferencia y recolección de métricas para la **Base 0**.

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = 'feat/19-base0-evaluation'

# --- Detectar entorno: Kaggle vs Colab ---
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

# 1. Clone repository sparsely from feature branch
if not REPO_PATH.exists():
    print(f"Clonando {REPO_NAME} (rama {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"El repositorio ya existe. Actualizando {REPO_NAME}...")
    %cd {REPO_NAME}
    !git checkout {BRANCH_NAME}
    !git pull -q origin {BRANCH_NAME}
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Fallo al navegar al directorio. Ruta actual: {current_dir}")

# 3. Install dependencies in editable mode
print("Instalando el paquete en modo editable con dependencias [cloud]...")
%pip install -q -e .[cloud]

## 2. Detección de Dataset y Configuración Incondicional del Token

In [1]:
import os
import sys
import time
from pathlib import Path
from src.inference.runners.run_base_0 import Base0Runner

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = os.path.exists('/content/drive')

# --- Búsqueda automática del dataset en Kaggle/Colab ---
def find_dataset_dir() -> Path:
    if not IS_KAGGLE:
        return Path('/content/drive/MyDrive/ia_article')
    
    # Probar rutas típicas de Kaggle
    candidates = [
        Path('/kaggle/input/datasets/alvaroquispeunsa/mtc-challenge'),
        Path('/kaggle/input/mtc-challenge'),
    ]
    for c in candidates:
        if (c / 'split_metadata.csv').exists():
            return c
    
    # Búsqueda recursiva en todo /kaggle/input
    for root, _, files in os.walk('/kaggle/input'):
        if 'split_metadata.csv' in files:
            return Path(root)
            
    # Fallback predeterminado
    return Path('/kaggle/input/mtc-challenge')

DATASET_DIR = find_dataset_dir()
OUTPUT_DIR = Path('/kaggle/working/output_base0') if IS_KAGGLE else Path('/content/output_base0')
TOKEN_PATH = Path('/kaggle/working/token.json') if IS_KAGGLE else Path('/content/token.json')

# --- Configuración del Runner ---
config = {
    "device": 0,
    "conf": 0.001,
    "iou": 0.45,
    "imgsz": 640,
    "batch_size": 16,
    "metadata_path": str(DATASET_DIR / "split_metadata.csv"),
    "images_dir": str(DATASET_DIR / "train-001" / "train"),
    "output_dir": str(OUTPUT_DIR),
    "hardware_name": "Tesla_T4_Kaggle" if IS_KAGGLE else "Colab_GPU",
    "experiment_condition": "Base_0_Zero_Shot",
    # Se pasa la ruta incondicionalmente para que IOManager lo descargue automáticamente si no existe
    "token_path": str(TOKEN_PATH),
    "drive_folder_id": "1wXieZvOZDE5KzZiYyESbf8xU-C2AGPWJ",
}

# --- Diagnóstico previo ---
metadata = Path(config["metadata_path"])
images = Path(config["images_dir"])
print(f"Entorno: {'Kaggle' if IS_KAGGLE else 'Colab'}")
print(f"Directorio dataset detectado: {DATASET_DIR}")
print(f"Metadata: {metadata} -> {'✅ existe' if metadata.exists() else '❌ NO EXISTE'}")
print(f"Imágenes: {images} -> {'✅ existe' if images.exists() else '❌ NO EXISTE'}")
print(f"Ruta destino token: {config['token_path']}")
print(f"Drive folder ID: {config['drive_folder_id']}")

if not metadata.exists() or not images.exists():
    raise FileNotFoundError(f"Rutas no encontradas. Verificó {metadata} e {images}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Entorno: Kaggle
Directorio dataset detectado: /kaggle/input/datasets/alvaroquispeunsa/mtc-challenge
Metadata: /kaggle/input/datasets/alvaroquispeunsa/mtc-challenge/split_metadata.csv -> ✅ existe
Imágenes: /kaggle/input/datasets/alvaroquispeunsa/mtc-challenge/train-001/train -> ✅ existe
Ruta destino token: /kaggle/working/token.json
Drive folder ID: 1wXieZvOZDE5KzZiYyESbf8xU-C2AGPWJ


In [2]:
# --- Ejecutar Inferencia ---
print("Inicializando ejecutor Base 0 (IOManager gestionará el token automáticamente)...")
runner = Base0Runner(config=config, model_path="yolo26m-obb.pt")

print("\nEjecutando inferencia (máx. 5 clips de prueba)...")
results = runner.execute()

print("\n" + "="*60)
print("RESULTADO FINAL")
print("="*60)
print(f"Estado: {results['status']}")
print(f"Archivos generados: {len(results['files'])}")
for f in results['files']:
    drive_status = f'Drive ID: {f["drive_id"]}' if f['drive_id'] else 'Solo local'
    print(f"  📄 {Path(f['local']).name} -> {drive_status}")
print(f"\nMétricas:")
for k, v in results['metrics'].items():
    print(f"  {k}: {v}")

Inicializando ejecutor Base 0 (IOManager gestionará el token automáticamente)...
[IOManager] token.json no encontrado en /kaggle/working/token.json. Descargando automáticamente desde Drive (ID: 1Fjg-AIrIQ77g1JRtapE6XDb_A6CP_4q3)...
[IOManager] ✅ token.json guardado en /kaggle/working/token.json
[Base0Runner] Mapeo DOTA -> MTC construido: {9: 6, 10: 0}

Ejecutando inferencia (máx. 5 clips de prueba)...
[Base0Runner] Clips a procesar (limitado a 5): ['v_01tp2eyell', 'v_03ah598sio', 'v_06xnbcrtte', 'v_09gpsv2mc8', 'v_0mp55gehz5']
  [1/5] v_01tp2eyell: 50 frames...
    ✅ Subido a Drive: v_01tp2eyell_predictions.json (id: 1SjNoDiUOT00IE6juencjkCpiadG-8r8M)
  [2/5] v_03ah598sio: 50 frames...
    ✅ Subido a Drive: v_03ah598sio_predictions.json (id: 1pVqNOmo_W85wMfOZb_2mAdjfPFoUNG_a)
  [3/5] v_06xnbcrtte: 50 frames...
    ✅ Subido a Drive: v_06xnbcrtte_predictions.json (id: 1laIHJk-qmp7Rzgi8PktyVsCMslpDEqTg)
  [4/5] v_09gpsv2mc8: 50 frames...
    ✅ Subido a Drive: v_09gpsv2mc8_predictions.json

In [ ]:
import os
# Esperar un par de segundos para asegurar que cualquier I/O pendiente (como la subida a Drive) finalice
import time
time.sleep(5)

# Matar todos los procesos del entorno
os.system("kill -9 -1")